# Build KEGG DB

Reads every sample's raw JGI files and writes the two prebuilt inputs
`EGC_KEGG_TREE.ipynb` reads:

- `kegg_db.json`: which gene has which KO, from `.a.ko.txt`.
- `ko_names.json`: one name per KO, from `.a.gene_product.txt` -- the
  product name JGI gives a gene when it named the gene from that KO.

Both come only from JGI's annotation files. KEGG's own hierarchy and KO
definitions (`ko00001.keg`) are not used: they are licensed content.

**Run order:** top to bottom. Re-run this whenever any sample's
`.a.ko.txt` or `.a.gene_product.txt` changes, before re-running
`EGC_KEGG_TREE.ipynb` -- that notebook checks `kegg_db.json`'s staleness
against the real raw files and refuses to run on an out-of-date one.

In [1]:
import json
from pathlib import Path

## 1. Read every sample's raw `.a.ko.txt` into `keggdb`

In [2]:
def read_ko(directorypath: Path, filename: str, sample: str, keggdb: dict) -> None:
    """
    Parse one sample's .a.ko.txt and append each gene's KO id(s) to keggdb.

    keggdb[gene_id] = {'lookupId': [...], 'timepoint': [...]} -- parallel
    lists, one entry per observation of that gene (a gene can legitimately
    have more than one KO hit).

    Direct path, not rglob -- there's exactly one real
    file per sample (confirmed live against all 128 samples), so searching
    the whole subtree for matches only risks silently reading a stray
    duplicate twice. No try/except around the read -- a real read
    failure should crash this notebook loudly, not print a message that
    scrolls past while the rest of the run continues on partial data.
    """
    file_path = directorypath / filename
    if not file_path.is_file():
        # Carrying on would leave this sample with no KEGG data at all.
        raise FileNotFoundError(f"No {filename} found in {directorypath}")

    with file_path.open() as f:
        content = f.readlines()
    for line in content:
        if not line.strip():
            continue
        info = line.strip().split('\t')
        # A gene line has its KO in the third column; skipping a damaged
        # line would quietly drop that gene's KO.
        if len(info) < 3:
            raise ValueError(f"{file_path}: fewer than 3 columns: {line[:80]!r}")

        gene = info[0]
        keggid = info[2]

        if gene not in keggdb:
            keggdb[gene] = {'lookupId': [keggid], 'timepoint': [sample]}
        else:
            keggdb[gene]['lookupId'].append(keggid)
            keggdb[gene]['timepoint'].append(sample)


with open(r'C:\Users\sakoehn\Desktop\JGI\sample_names.txt') as f:
    samples = [s.strip() for s in f]
print(f"Loaded {len(samples)} samples")

base = Path(r"C:\Users\sakoehn\Desktop\JGI\Metagenome_Samples")
keggdb: dict = {}

for s in samples:
    read_ko(base / s, f"{s}.a.ko.txt", s, keggdb)

print(f"Total entries in keggdb: {len(keggdb):,}")

Loaded 128 samples


Total entries in keggdb: 36,803,002


## 2. Write `kegg_db.json`

In [3]:
kegg_db_path = Path(r"C:\Users\sakoehn\Desktop\JGI\DataStructures\Kegg\kegg_db.json")
kegg_db_path.parent.mkdir(parents=True, exist_ok=True)

with open(kegg_db_path, 'w') as f:
    json.dump(keggdb, f)

print(f"Wrote {kegg_db_path} ({len(keggdb):,} genes)")

Wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Kegg\kegg_db.json (36,803,002 genes)


## 3. KO names from `.a.gene_product.txt` → `ko_names.json`

Each `.a.gene_product.txt` line is `gene_id, product name, source`. When the
source is a KO (`KO:K00001`), the name is JGI's name for that KO. A KO is
given exactly one name across all 128 samples, so a second, different name
for the same KO means the files changed and stops the run rather than
picking one.

KOs whose genes JGI always named from another source (a COG, say) get no
entry here and are shown by their id.

In [4]:
ko_names: dict = {}
for s in samples:
    path = base / s / f"{s}.a.gene_product.txt"
    if not path.is_file():
        raise FileNotFoundError(f"No {path.name} found in {path.parent}")
    with path.open(encoding="utf-8") as f:
        for line in f:
            info = line.rstrip("\n").split("\t")
            if len(info) < 3 or not info[2].startswith("KO:K"):
                continue
            ko, name = info[2][3:], info[1]
            if ko_names.setdefault(ko, name) != name:
                raise ValueError(
                    f"{path}: {ko} named {name!r}, but {ko_names[ko]!r} elsewhere"
                )

ko_names_path = kegg_db_path.parent / "ko_names.json"
with open(ko_names_path, "w", encoding="utf-8") as f:
    json.dump(dict(sorted(ko_names.items())), f, ensure_ascii=False, indent=0)
print(f"Wrote {ko_names_path} ({len(ko_names):,} KO names)")

Wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Kegg\ko_names.json (10,069 KO names)
